In [ ]:
# ================================================
# SETUP COLAB — Ejecutar primero
# El CSV dataset_preprocessed.csv debe estar en /content/
# ================================================
!pip install -q transformers torch datasets scikit-learn


In [ ]:
# ================================================
# Notebook 04: BERT Fine-tuning
# Autor: Steeven Quezada
# Entorno: Google Colab GPU T4
# ================================================

import pandas as pd
import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

# ================================================
# CONFIGURACIÓN PRINCIPAL
# ================================================
MODEL_NAME = 'bert-base-uncased'
# Para prueba rápida puedes usar:
# MODEL_NAME = 'distilbert-base-uncased'

# Detectar dispositivo disponible
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Modelo seleccionado: {MODEL_NAME}')
print(f'Dispositivo:         {DEVICE}')
print(f'GPU disponible:      {torch.cuda.is_available()}')

# ================================================
# PASO 1: CARGA DE DATOS
# ================================================
df = pd.read_csv('dataset_preprocessed.csv', low_memory=False)
print(f'\nDataset cargado: {len(df):,} registros')
print(f'Distribución: Falsas {(df["label"]==0).sum():,} | Reales {(df["label"]==1).sum():,}')

Modelo seleccionado: bert-base-uncased
Dispositivo:         cuda
GPU disponible:      True

Dataset cargado: 38,473 registros
Distribución: Falsas 17,282 | Reales 21,191


In [3]:
# ================================================
# PASO 2: DIVISIÓN TRAIN / VALIDATION / TEST
# ================================================
from sklearn.model_selection import train_test_split

# Usamos el enfoque ganador del baseline: título + texto
X = df['title_text_clean'].fillna('')
y = df['label']

# Primera división: 85% train+val / 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Segunda división: 70% train / 15% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,  # 0.176 * 0.85 ≈ 0.15 del total
    random_state=42,
    stratify=y_temp
)

print('=== DIVISIÓN DE DATOS ===')
print(f'Train:      {len(X_train):,} ({len(X_train)/len(df)*100:.1f}%)')
print(f'Validation: {len(X_val):,} ({len(X_val)/len(df)*100:.1f}%)')
print(f'Test:       {len(X_test):,} ({len(X_test)/len(df)*100:.1f}%)')
print(f'Total:      {len(df):,}')
print(f'\nBalance Train — Falsas: {(y_train==0).sum():,} | Reales: {(y_train==1).sum():,}')
print(f'Balance Val   — Falsas: {(y_val==0).sum():,} | Reales: {(y_val==1).sum():,}')
print(f'Balance Test  — Falsas: {(y_test==0).sum():,} | Reales: {(y_test==1).sum():,}')

=== DIVISIÓN DE DATOS ===
Train:      26,946 (70.0%)
Validation: 5,756 (15.0%)
Test:       5,771 (15.0%)
Total:      38,473

Balance Train — Falsas: 12,104 | Reales: 14,842
Balance Val   — Falsas: 2,586 | Reales: 3,170
Balance Test  — Falsas: 2,592 | Reales: 3,179


In [4]:
# ================================================
# PASO 3: TOKENIZACIÓN
# ================================================
from transformers import AutoTokenizer

print(f'Cargando tokenizador: {MODEL_NAME}')
print('Descargando modelo la primera vez...\n')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Longitud máxima de tokens
# BERT acepta hasta 512 — usamos 256 para reducir memoria y tiempo
MAX_LENGTH = 256

def tokenizar(textos, max_length=MAX_LENGTH):
    return tokenizer(
        list(textos),
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors='pt'
    )

print('Tokenizando conjuntos...')
train_encodings = tokenizar(X_train)
val_encodings   = tokenizar(X_val)
test_encodings  = tokenizar(X_test)

print(f'✓ Train tokenizado:      {train_encodings["input_ids"].shape}')
print(f'✓ Validation tokenizado: {val_encodings["input_ids"].shape}')
print(f'✓ Test tokenizado:       {test_encodings["input_ids"].shape}')
print(f'\nMax length usado: {MAX_LENGTH} tokens')

Cargando tokenizador: bert-base-uncased
Descargando modelo la primera vez...



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizando conjuntos...
✓ Train tokenizado:      torch.Size([26946, 256])
✓ Validation tokenizado: torch.Size([5756, 256])
✓ Test tokenizado:       torch.Size([5771, 256])

Max length usado: 256 tokens


In [5]:
# ================================================
# PASO 4: DATASET DE PYTORCH
# ================================================
from torch.utils.data import Dataset, DataLoader

class FakeNewsDataset(Dataset):
    """
    Dataset personalizado para PyTorch.
    Convierte los encodings y etiquetas en
    formato que el modelo puede consumir.
    """
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels.reset_index(drop=True)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: val[idx].clone().detach()
            for key, val in self.encodings.items()
        }
        item['labels'] = torch.tensor(
            self.labels[idx], dtype=torch.long
        )
        return item

# Crear datasets
train_dataset = FakeNewsDataset(train_encodings, y_train)
val_dataset   = FakeNewsDataset(val_encodings,   y_val)
test_dataset  = FakeNewsDataset(test_encodings,  y_test)

# batch_size=32 para GPU T4
# Si hay errores de memoria baja a 16
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print('=== DATALOADERS CREADOS ===')
print(f'Batch size:        {BATCH_SIZE}')
print(f'Batches en train:  {len(train_loader):,}')
print(f'Batches en val:    {len(val_loader):,}')
print(f'Batches en test:   {len(test_loader):,}')
print(f'\n✓ Dataset listo para entrenamiento')

=== DATALOADERS CREADOS ===
Batch size:        32
Batches en train:  843
Batches en val:    180
Batches en test:   181

✓ Dataset listo para entrenamiento


In [6]:
# ================================================
# PASO 5: CARGAR MODELO BERT
# ================================================
from transformers import AutoModelForSequenceClassification

print(f'Cargando modelo: {MODEL_NAME}')
print('Descargando pesos preentrenados...\n')

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # binario: falsa (0) o real (1)
)

model = model.to(DEVICE)

# Contar parámetros
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✓ Modelo cargado en: {DEVICE}')
print(f'  Parámetros totales:     {total_params:,}')
print(f'  Parámetros entrenables: {trainable_params:,}')

Cargando modelo: bert-base-uncased
Descargando pesos preentrenados...



model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Modelo cargado en: cuda
  Parámetros totales:     109,483,778
  Parámetros entrenables: 109,483,778


In [7]:
# ================================================
# PASO 6: CONFIGURACIÓN DEL ENTRENAMIENTO
# ================================================
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Hiperparámetros
# PRUEBA en Colab T4:    EPOCHS = 1
# DEFINITIVO en GPU U:   EPOCHS = 3
EPOCHS        = 3      # ← cambiar a 3 para entrenamiento final
LEARNING_RATE = 2e-5   # recomendado en el paper original de BERT
WARMUP_STEPS  = 100    # pasos de calentamiento del learning rate

# Optimizador
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01  # regularización L2
)

# Scheduler — reduce el learning rate gradualmente
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

print('=== CONFIGURACIÓN DE ENTRENAMIENTO ===')
print(f'Épocas:           {EPOCHS}')
print(f'Learning rate:    {LEARNING_RATE}')
print(f'Batch size:       {BATCH_SIZE}')
print(f'Pasos totales:    {total_steps:,}')
print(f'Warmup steps:     {WARMUP_STEPS}')
print(f'Optimizador:      AdamW (torch.optim)')
print(f'Scheduler:        Linear warmup + decay')
print(f'\nEstimación GPU T4 (1 época): ~15-20 minutos')
print(f'Estimación GPU U  (3 épocas): ~45-60 minutos')

=== CONFIGURACIÓN DE ENTRENAMIENTO ===
Épocas:           3
Learning rate:    2e-05
Batch size:       32
Pasos totales:    2,529
Warmup steps:     100
Optimizador:      AdamW (torch.optim)
Scheduler:        Linear warmup + decay

Estimación GPU T4 (1 época): ~15-20 minutos
Estimación GPU U  (3 épocas): ~45-60 minutos


In [8]:
# ================================================
# PASO 7: ENTRENAMIENTO
# ================================================
from sklearn.metrics import accuracy_score, f1_score
import time

def evaluar(model, loader, device):
    """Evalúa el modelo en un conjunto de datos."""
    model.eval()
    preds_all, labels_all = [], []
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(labels_all, preds_all)
    f1       = f1_score(labels_all, preds_all, average='weighted')
    return avg_loss, acc, f1


# ── Loop de entrenamiento ──────────────────────
print('=== INICIANDO ENTRENAMIENTO ===')
print(f'Modelo: {MODEL_NAME} en {DEVICE}\n')

historial = []
inicio_total = time.time()

for epoca in range(EPOCHS):
    model.train()
    total_loss_train = 0
    inicio_epoca = time.time()

    for i, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        # Gradient clipping — evita gradientes explosivos
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss_train += loss.item()

        # Mostrar progreso cada 50 batches
        if (i + 1) % 50 == 0:
            elapsed = time.time() - inicio_epoca
            pct     = (i + 1) / len(train_loader) * 100
            print(f'  Época {epoca+1} | Batch {i+1}/{len(train_loader)} '
                  f'({pct:.0f}%) | Loss: {loss.item():.4f} | '
                  f'Tiempo: {elapsed/60:.1f} min')

    # Evaluar en validación al final de cada época
    avg_train_loss = total_loss_train / len(train_loader)
    val_loss, val_acc, val_f1 = evaluar(model, val_loader, DEVICE)
    tiempo_epoca = (time.time() - inicio_epoca) / 60

    historial.append({
        'epoca':      epoca + 1,
        'train_loss': avg_train_loss,
        'val_loss':   val_loss,
        'val_acc':    val_acc,
        'val_f1':     val_f1,
        'tiempo_min': tiempo_epoca
    })

    print(f"\n{'='*55}")
    print(f'ÉPOCA {epoca+1}/{EPOCHS} completada en {tiempo_epoca:.1f} min')
    print(f'  Train Loss: {avg_train_loss:.4f}')
    print(f'  Val Loss:   {val_loss:.4f}')
    print(f'  Val Acc:    {val_acc:.4f}')
    print(f'  Val F1:     {val_f1:.4f}')
    print(f"{'='*55}\n")

tiempo_total = (time.time() - inicio_total) / 60
print(f'Entrenamiento completado en {tiempo_total:.1f} minutos')

=== INICIANDO ENTRENAMIENTO ===
Modelo: bert-base-uncased en cuda

  Época 1 | Batch 50/843 (6%) | Loss: 0.4197 | Tiempo: 1.1 min
  Época 1 | Batch 100/843 (12%) | Loss: 0.0254 | Tiempo: 2.2 min
  Época 1 | Batch 150/843 (18%) | Loss: 0.0491 | Tiempo: 3.4 min
  Época 1 | Batch 200/843 (24%) | Loss: 0.0023 | Tiempo: 4.6 min
  Época 1 | Batch 250/843 (30%) | Loss: 0.0020 | Tiempo: 5.8 min
  Época 1 | Batch 300/843 (36%) | Loss: 0.0090 | Tiempo: 7.0 min
  Época 1 | Batch 350/843 (42%) | Loss: 0.0004 | Tiempo: 8.2 min
  Época 1 | Batch 400/843 (47%) | Loss: 0.0003 | Tiempo: 9.3 min
  Época 1 | Batch 450/843 (53%) | Loss: 0.0003 | Tiempo: 10.5 min
  Época 1 | Batch 500/843 (59%) | Loss: 0.0007 | Tiempo: 11.7 min
  Época 1 | Batch 550/843 (65%) | Loss: 0.0002 | Tiempo: 12.9 min
  Época 1 | Batch 600/843 (71%) | Loss: 0.0003 | Tiempo: 14.1 min
  Época 1 | Batch 650/843 (77%) | Loss: 0.0001 | Tiempo: 15.3 min
  Época 1 | Batch 700/843 (83%) | Loss: 0.0002 | Tiempo: 16.5 min
  Época 1 | Batch 7

In [9]:
# ================================================
# PASO 8: GUARDAR MODELO ENTRENADO
# ================================================
import os, json

os.makedirs('bert_finetuned', exist_ok=True)

# Guardar modelo y tokenizador
model.save_pretrained('bert_finetuned/')
tokenizer.save_pretrained('bert_finetuned/')

# Guardar historial
with open('bert_finetuned/historial.json', 'w') as f:
    json.dump(historial, f, indent=2)

print('✓ Modelo guardado en bert_finetuned/')
print('✓ Historial guardado en bert_finetuned/historial.json')

# Descargar carpeta como zip
!zip -r bert_finetuned.zip bert_finetuned/

from google.colab import files
files.download('bert_finetuned.zip')
print('\n✓ Descarga iniciada — guarda el zip en tu carpeta models/')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Modelo guardado en bert_finetuned/
✓ Historial guardado en bert_finetuned/historial.json
  adding: bert_finetuned/ (stored 0%)
  adding: bert_finetuned/model.safetensors (deflated 7%)
  adding: bert_finetuned/config.json (deflated 53%)
  adding: bert_finetuned/tokenizer_config.json (deflated 43%)
  adding: bert_finetuned/historial.json (deflated 60%)
  adding: bert_finetuned/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Descarga iniciada — guarda el zip en tu carpeta models/


In [10]:
# ================================================
# PASO 9: EVALUACIÓN FINAL EN TEST
# ================================================
print('=== EVALUACIÓN FINAL EN CONJUNTO DE TEST ===')
test_loss, test_acc, test_f1 = evaluar(model, test_loader, DEVICE)

print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Test F1-score: {test_f1:.4f}')
print(f'\nComparación con baseline SVM:')
print(f'  SVM Accuracy:  99.45%')
print(f'  BERT Accuracy: {test_acc*100:.2f}%')
print(f'  Diferencia:    {(test_acc - 0.9945)*100:+.2f} puntos')

=== EVALUACIÓN FINAL EN CONJUNTO DE TEST ===
Test Loss:     0.0066
Test Accuracy: 0.9991 (99.91%)
Test F1-score: 0.9991

Comparación con baseline SVM:
  SVM Accuracy:  99.45%
  BERT Accuracy: 99.91%
  Diferencia:    +0.46 puntos
